# Create structures template and dataset
Full pipeline: build `template_100m_structures.pkl` (weirs/thin dams embedded as constrained edges in the coarse meshes, SFINCS grid as the finest scale) → convert each levee/dam simulation onto it → merge the training ones (plus the existing BC-augmentation multisim events) into one train pkl.

In [ ]:
import os, sys

# Resolve the repo root robustly (works in VS Code and nbconvert, any start cwd)
try:
    REPO_ROOT = os.path.abspath(os.path.join(os.path.dirname(__vsc_ipynb_file__), '..'))
except NameError:
    _here = os.getcwd()
    REPO_ROOT = _here if os.path.isdir(os.path.join(_here, 'database')) \
                else os.path.abspath(os.path.join(_here, '..'))
assert os.path.isdir(os.path.join(REPO_ROOT, 'database')), f'Not the repo root: {REPO_ROOT}'
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print('Repo root:', REPO_ROOT)

## Config

In [ ]:
# Template output path
TEMPLATE_PKL = 'database/datasets/train/template_100m_structures.pkl'

# Base (structure-free) simulation used for the boundary polygon, DEM, and finest-scale
# SFINCS grid — grid geometry is identical across every simulation in this catchment,
# structures are overlaid via sfincs.weir on top of the same grid, not a different mesh.
SFINCS_DIR = (
    'database/raw_datasets_ahr/Simulations/'
    'ahr_river_v03_Marg_additionalsrc_velocity_100m_cutpolygon'
)
SFINCS_MAP_TEMPLATE = os.path.join(SFINCS_DIR, 'sfincs_map.nc')
SFINCS_MAP_GRID     = os.path.join(SFINCS_DIR, 'sfincs_map.nc')
SHAPEFILE = os.path.join(SFINCS_DIR, 'gis', 'region.geojson')
DEM_TIF   = os.path.join(SFINCS_DIR, 'gis', 'dep.tif')

# Merged weir/thin-dam line file (EPSG:32632, one LineString per structure, no buffering —
# create_gmesh embeds LineStrings directly as constrained edges and simplifies them to
# match each coarse scale's resolution before embedding).
STRUCTURES_GEOJSON = os.path.join(SFINCS_DIR, 'gis', 'structures.geojson')

# Coarse gmesh resolutions [m] — 3 levels, coarsest to finest
MESH_RESOLUTIONS = [2000, 1000, 500]

# Simulations to convert onto the structures template
SIM_ROOT = 'database/raw_datasets_ahr/Simulations'
SIM_FOLDER = 'ahr_river_v03_Marg_additionalsrc_velocity_100m_cutpolygon_warmstart_infra_{tag}'

TRAIN_SIMS = {
    tag: SIM_FOLDER.format(tag=tag) for tag in [
        'bothbanks_dz2m', 'bothbanks_dz5m',
        'northonly_dz2m', 'northonly_dz5m',
        'southonly_dz2m', 'southonly_dz5m',
        'upstream_dz2m', 'upstream_dz5m',
        'downstream_dz2m', 'downstream_dz5m',
        'thin_dam_dz15m', 'thin_dam_dz20m',
    ]
}

TEST_SIMS = {
    tag: SIM_FOLDER.format(tag=tag) for tag in [
        'bothbanks_dz10m',    # unseen magnitude (2x beyond the highest trained levee crest)
        'gap_dz5m',            # unseen structural footprint (a breach in the trained levee family)
        'thin_dam2_dz15m',     # unseen location, same thin-dam mechanism (~3-4 km away)
        'thin_dam2_dz20m',
    ]
}

# Already-converted BC-augmentation multisim training pkl (20 events, on template_100m.pkl,
# not this structures template — merged in below anyway, each Data object carries its own
# mesh/edge_index so mixing templates within one training list is fine).
EXISTING_TRAIN_PKL = 'database/datasets/train/ahr_river_v03_marg_bcaugment_additionalsrc_velocity_100m_warmstart_multisim.pkl'

PER_SIM_NAME = 'ahr_river_v03_marg_structures_{tag}_additionalsrc_velocity_100m_warmstart'
MERGED_NAME  = 'ahr_river_v03_marg_structures_bcaugment_additionalsrc_velocity_100m_warmstart_multisim'
OUT_ROOT = 'database/datasets'

# SFINCS variable names
WATER_LEVEL_VAR = 'zs'
BED_LEVEL_VAR   = 'zb'
VX_VAR = 'u'
VY_VAR = 'v'

# Force flags
FORCE_REBUILD_TEMPLATE = False   # set True to recreate template from scratch
FORCE_REBUILD_PKL      = False   # set True to reconvert simulations already on disk

## Step 1 — Create template (with structures embedded)

In [ ]:
# import as a bare top-level module (database/ on sys.path directly), NOT as
# database.create_mesh_template_marg — the latter runs database/__init__.py first,
# which imports torch immediately and breaks the gmsh-subprocess ordering that
# create_mesh_template_pkl relies on to build coarse meshes reliably.
sys.path.insert(0, os.path.join(REPO_ROOT, 'database'))
from create_mesh_template_marg import create_mesh_template_pkl

if FORCE_REBUILD_TEMPLATE and os.path.exists(TEMPLATE_PKL):
    os.remove(TEMPLATE_PKL)
    print('Deleted stale template:', TEMPLATE_PKL)

if os.path.exists(TEMPLATE_PKL):
    print('Template already exists:', TEMPLATE_PKL)
    print('Set FORCE_REBUILD_TEMPLATE = True to recreate.')
else:
    create_mesh_template_pkl(
        shapefile_path        = SHAPEFILE,
        dem_tif_path          = DEM_TIF,
        output_pkl_path       = TEMPLATE_PKL,
        with_multiscale       = True,
        number_of_multiscales = 4,
        mesh_resolutions      = MESH_RESOLUTIONS,
        sfincs_map_nc         = SFINCS_MAP_TEMPLATE,
        structures_gpkg_path  = STRUCTURES_GEOJSON,
    )
    print('Template saved:', TEMPLATE_PKL)

In [ ]:
import matplotlib.pyplot as plt
from database.graph_creation import plot_faces
from database.convert_sfincs_to_pkl_marg import load_single_data_object

template_check = load_single_data_object(TEMPLATE_PKL)
meshes = template_check.mesh.meshes
n = len(meshes)

fig, axs = plt.subplots(1, n, figsize=(n * 5, 5))
fig.suptitle('Mesh levels: coarsest → finest (with structures embedded)', fontsize=13)

for i, m in enumerate(meshes):
    ax = axs[i] if n > 1 else axs
    lw = max(0.05, 0.35 - i * 0.07)
    plot_faces(m, ax=ax, facecolor='none', edgecolor='black', linewidth=lw)
    ax.set_aspect('equal')
    ax.set_title(f'Level {i}  |  {m.face_x.shape[0]:,} faces', fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout()
plt.show()

## Step 2 — Convert each simulation onto the template

In [ ]:
import pickle
import numpy as np
import torch
import xarray as xr

from database.convert_sfincs_to_pkl_marg import (
    load_single_data_object,
    get_target_points,
    get_source_points,
    interpolate_time_series,
    parse_src_file,
    parse_dis_file,
    build_output_data,
)

print('Loading template...')
template_data = load_single_data_object(TEMPLATE_PKL)
target_points = get_target_points(template_data)
print('  Template mesh faces:', target_points.shape[0])

ds_grid = xr.open_dataset(SFINCS_MAP_GRID, decode_times=False)
source_points = get_source_points(ds_grid)
ds_grid.close()
print('  Source points:', source_points.shape[0])

all_sims = {**TRAIN_SIMS, **TEST_SIMS}
sim_split = {tag: 'train' for tag in TRAIN_SIMS}
sim_split.update({tag: 'test' for tag in TEST_SIMS})

for tag, folder in all_sims.items():
    dataset_name = PER_SIM_NAME.format(tag=tag)
    out_split = sim_split[tag]
    out_path = os.path.join(OUT_ROOT, out_split, dataset_name + '.pkl')
    if not FORCE_REBUILD_PKL and os.path.exists(out_path):
        print('Skipping (already exists):', out_path)
        continue

    sim_dir = os.path.join(SIM_ROOT, folder)
    sfincs_map = os.path.join(sim_dir, 'sfincs_map.nc')
    src_file   = os.path.join(sim_dir, 'sfincs.src')
    dis_file   = os.path.join(sim_dir, 'sfincs.dis')

    print()
    print('Processing:', dataset_name, '->', out_split)
    ds = xr.open_dataset(sfincs_map, decode_times=False)

    zs = ds[WATER_LEVEL_VAR].values
    zb = ds[BED_LEVEL_VAR].values
    # SFINCS writes zs=NaN for dry cells - fill with bed level so they enter the
    # interpolation as WD=0 (see create_dataset_100m.ipynb for why this matters).
    zs_filled = np.where(np.isnan(zs), zb[None, :, :], zs)
    WD_grid = np.maximum(zs_filled - zb[None, :, :], 0.0).astype(np.float32)
    print('  Interpolating WD...')
    WD = interpolate_time_series(source_points, WD_grid, target_points, 'WD')

    ds_raw = xr.open_dataset(sfincs_map, decode_times=False, mask_and_scale=False)
    if VX_VAR and VX_VAR in ds.data_vars:
        VX_raw = ds_raw[VX_VAR].values.astype(np.float32)
        fv = ds_raw[VX_VAR].attrs.get('_FillValue', None)
        if fv is not None:
            VX_raw[VX_raw == fv] = np.nan
        print('  Interpolating VX...')
        VX = interpolate_time_series(source_points, VX_raw, target_points, 'VX')
    else:
        VX = np.zeros_like(WD)
    if VY_VAR and VY_VAR in ds.data_vars:
        VY_raw = ds_raw[VY_VAR].values.astype(np.float32)
        fv = ds_raw[VY_VAR].attrs.get('_FillValue', None)
        if fv is not None:
            VY_raw[VY_raw == fv] = np.nan
        print('  Interpolating VY...')
        VY = interpolate_time_series(source_points, VY_raw, target_points, 'VY')
    else:
        VY = np.zeros_like(WD)
    ds_raw.close()

    time_var = ds.coords.get('time', ds.coords.get('t', None))
    map_times_s = (time_var.values.astype(np.float64) if time_var is not None
                   else np.arange(zs.shape[0]) * 3600.0)
    ds.close()

    print('  Reading src/dis files...')
    src_xy = parse_src_file(src_file)
    dis_times_s, discharge = parse_dis_file(dis_file)
    print(' ', len(src_xy), 'source points, discharge shape:', discharge.shape)

    data_out = build_output_data(
        template_data, WD=WD, VX=VX, VY=VY,
        map_times_s=map_times_s, src_xy=src_xy,
        dis_times_s=dis_times_s, discharge=discharge,
    )

    os.makedirs(os.path.join(OUT_ROOT, out_split), exist_ok=True)
    with open(out_path, 'wb') as f:
        pickle.dump([data_out], f)
    print('  Saved:', out_path)
    print('  WD=', tuple(data_out.WD.shape),
          '| node_BC=', data_out.node_BC.tolist(),
          '| BC=', tuple(data_out.BC.shape))

    np_ptr = data_out.node_ptr.numpy()
    wd_s0 = data_out.WD[np_ptr[0]:np_ptr[1]].numpy()
    wet = (wd_s0 > 0.05).sum(0)
    print(f'  wet cells @0.05m: t=0 {wet[0]}  peak {wet.max()} (t={wet.argmax()})  t=-1 {wet[-1]}')

## Step 3 — Merge training simulations (+ existing BC-augmentation events) into one pkl
`TEST_SIMS` stay as separate per-scenario pkls — evaluate them individually for the three generalization checks (magnitude, footprint, location).

In [ ]:
merged = []
for tag in TRAIN_SIMS:
    pkl_path = os.path.join(OUT_ROOT, 'train', PER_SIM_NAME.format(tag=tag) + '.pkl')
    with open(pkl_path, 'rb') as f:
        data_list = pickle.load(f)
    for data in data_list:
        print(f'{tag}: node_BC = {data.node_BC.tolist()}   WD peak = {float(data.WD.max()):.3f} m   '
              f'n_steps = {data.WD.shape[1]}')
    merged += data_list

assert os.path.exists(EXISTING_TRAIN_PKL), \
    f'Existing BC-augmentation multisim pkl not found: {EXISTING_TRAIN_PKL} -- run run_convert_bc_augmentation.py first'
with open(EXISTING_TRAIN_PKL, 'rb') as f:
    existing = pickle.load(f)
print(f'\nMerging in {len(existing)} existing BC-augmentation events from {EXISTING_TRAIN_PKL}')
merged += existing

out_path = os.path.join(OUT_ROOT, 'train', MERGED_NAME + '.pkl')
with open(out_path, 'wb') as f:
    pickle.dump(merged, f)
print(f'\nSaved {len(merged)} training events '
      f'({len(TRAIN_SIMS)} structure scenarios + {len(existing)} existing BC-augmentation events) -> {out_path}')